# Notebook 04 – Stable-Baselines3 Wrapper Test

### Ziel des Notebooks:

In diesem Notebook wird geprüft, ob das Unity-Environment technisch sauber mit Stable-Baselines3 verbunden werden kann. Das ist notwendig, weil PPO direkt über Unity ML-Agents trainiert werden kann, A2C und DQN aber über Stable-Baselines3 umgesetzt werden.

Der Fokus liegt in diesem Notebook noch nicht auf finaler Performance. Stattdessen soll geprüft werden, ob die technische Grundlage für spätere Trainingsläufe funktioniert:

- Verbindung zwischen Python und Unity Editor
- Übergabe des Unity-Environments an Stable-Baselines3
- korrekter Action Space als `Discrete(9)`
- Verarbeitung der Observations als flacher Vektor
- kurzer A2C-Testlauf
- kurzer DQN-Testlauf
- Speichern der Testmodelle

Dieses Notebook ist damit ein technischer Zwischenschritt vor dem Hyperparameter-Tuning und den finalen Trainingsruns.

---

### Einordnung im Projekt

Bis zu diesem Punkt wurden die Unity-Environments aufgebaut und die Random Baseline ausgewertet. Die Random Baseline dient als untere Referenz, um später einordnen zu können, ob die lernenden Algorithmen tatsächlich besser als zufälliges Verhalten abschneiden.

Als nächster Schritt wird geprüft, ob A2C und DQN technisch mit dem Unity-Environment trainiert werden können. Beide Algorithmen werden nicht direkt über Unity ML-Agents trainiert, sondern über Stable-Baselines3. Dafür braucht es eine stabile Schnittstelle zwischen Unity und Python.

PPO wird in diesem Projekt separat über Unity ML-Agents trainiert, da PPO dort nativ unterstützt wird.

---

## Technische Schnittstelle zwischen Unity und Stable-Baselines3

Unity ML-Agents stellt mit `UnityToGymWrapper` eine Möglichkeit bereit, ein Unity-Environment aus Python heraus wie ein Gym-Environment zu verwenden. Für Stable-Baselines3 reicht eine einfache Übergabe aber nicht immer aus, weil Stable-Baselines3 in aktuellen Versionen stärker auf Gymnasium-kompatible Environments ausgelegt ist.

Zusätzlich nutzt der Agent in Unity mehrere Informationsquellen:

- manuell definierte Vector Observations, z. B. Richtung und Distanz zum Ziel,
- Ray Perception Sensoren zur Wahrnehmung von Wänden und Zielen.

Für A2C und DQN ist es am einfachsten und stabilsten, wenn diese Informationen als ein flacher Vektor an das neuronale Netzwerk übergeben werden. Deshalb wird in diesem Notebook ein kleiner Adapter-Wrapper genutzt. Dieser Wrapper verändert nicht das Unity-Environment selbst, sondern nur die Python-Schnittstelle zwischen Unity und Stable-Baselines3.

Die Umgebung, die Rewards, die Bewegungslogik und der Action Space bleiben unverändert.

---

## Unity-Setup vor dem Start

Bevor die Verbindung aus Python gestartet wird, muss in Unity `Env1_Simple` geöffnet sein.

Checkliste für den `MazeAgent`:

- `MazeAgentScript` ist aktiv
- `RandomAgentScript` ist entfernt oder deaktiviert
- `EpisodeLogger` ist deaktiviert oder `enableLogging = false`
- Behavior Type = `Default`
- Behavior Name = `MazeAgent`
- Model = `None`
- Vector Observation Space Size = `4`
- Discrete Branches = `1`
- Branch 0 Size = `9`
- Decision Requester ist aktiv
- Decision Period = `1`
- Take Actions Between Decisions ist aktiviert
- Max Step = `5000`

Wichtig: Bei den Verbindungsschritten wird zuerst die Python-Zelle gestartet und danach in Unity auf Play gedrückt.

---

### Imports 

In dieser Codezelle werden alle Bibliotheken importiert, die für den Wrapper-Test benötigt werden. 
Dazu gehören die Unity-ML-Agents-Umgebung, der Gym-Wrapper und die beiden Stable-Baselines3-Algorithmen A2C und DQN

In [1]:
from pathlib import Path  # wird genutzt, um Pfade sauber zu verwalten
import sys  # wird genutzt, um den Projektroot für eigene Imports verfügbar zu machen
import os  # wird genutzt, um das Arbeitsverzeichnis auf den Projektroot zu setzen
import random  # wird für Python-Seeds genutzt
from typing import Any, Optional  # Optional ist für Python 3.9 kompatibel

import numpy as np  # wird für numerische Operationen genutzt
import torch  # Stable-Baselines3 basiert auf PyTorch
import gymnasium as gym  # Gymnasium-Interface für Stable-Baselines3
from gymnasium import spaces  # wird für Observation Space und Action Space genutzt

PROJECT_ROOT_NOTEBOOK = Path("..").resolve()  # Projektroot aus Sicht des Notebooks
sys.path.insert(0, str(PROJECT_ROOT_NOTEBOOK))  # Projektroot für eigene Imports verfügbar machen

from lib_py.paths import (  # zentrale Projektpfade aus eigener Hilfsdatei importieren
    PROJECT_ROOT,
    PROJECT_NAME,
    MAZE_AGENT_DIR,
    TRAINING_DIR,
    MODEL_DIR,
    LOG_DIR,
    ensure_project_dirs,
    show_project_path,
)

from mlagents_envs.environment import UnityEnvironment  # verbindet Python mit Unity
from mlagents_envs.envs.unity_gym_env import UnityToGymWrapper  # macht Unity zunächst gym-kompatibel

from stable_baselines3 import A2C, DQN  # A2C und DQN aus Stable-Baselines3

os.chdir(PROJECT_ROOT)  # wichtig: relative Pfade beziehen sich ab jetzt auf den Projektroot

print("Imports funktionieren.")  # kurzer Check

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Imports funktionieren.


---

### Projektpfade prüfen

Die zentralen Projektpfade werden aus `lib_py.paths` geladen. Die Ausgabe zeigt die Pfade nur relativ ab dem Projektordner, damit keine lokalen Windows-Pfade im Notebook gespeichert werden.

In [2]:
ensure_project_dirs()  # wichtige Projektordner erstellen, falls sie noch fehlen

print("Projektpfade geladen.")  # kurzer Check
print("Project Root:", PROJECT_NAME)  # nur Projektname anzeigen
print("Maze Agent Dir:", show_project_path(MAZE_AGENT_DIR))  # Unity-Projektordner relativ anzeigen
print("Model Dir:", show_project_path(MODEL_DIR))  # Modellordner relativ anzeigen
print("Log Dir:", show_project_path(LOG_DIR))  # Logordner relativ anzeigen

Projektpfade geladen.
Project Root: rl_navigation_project
Maze Agent Dir: rl_navigation_project/MazeAgent
Model Dir: rl_navigation_project/training/models
Log Dir: rl_navigation_project/training/logs


---

### Testkonfiguration

In dieser Coedzelle werden die Parameter für den technischen Wrapper-Test gesetzt. Die Timesteps bleiben bewusst niedrig, weil es hier noch nicht um finale Trainingsläufe geht. Ziel ist nur, die Verbindung zwischen Unity und Stable-Baselines3 zu prüfen.

In [3]:
SEED = 1  # erster technischer Testseed
ENV_NAME = "Env1_Simple"  # Wrapper-Test wird zuerst auf Env1 gemacht
TEST_TIMESTEPS = 10_000  # kurzer Testlauf, noch kein finales Training

print("Testkonfiguration gesetzt.")  # kurzer Check
print("Seed:", SEED)  # Seed anzeigen
print("Environment:", ENV_NAME)  # Environment anzeigen
print("Test Timesteps:", TEST_TIMESTEPS)  # Testlänge anzeigen

Testkonfiguration gesetzt.
Seed: 1
Environment: Env1_Simple
Test Timesteps: 10000


---

### Seeds setzen

Damit die Python-Seite reproduzierbarer läuft, werden die Seeds für Python, NumPy und PyTorch gesetzt. Der Unity-Seed wird zusätzlich beim Start des Unity-Environments übergeben und im Unity-Projekt über den `SeedManager` (C#-Script) verwaltet.

In [4]:
def set_global_seeds(seed: int) -> None:
    random.seed(seed)  # Python-Zufall setzen
    np.random.seed(seed)  # NumPy-Zufall setzen
    torch.manual_seed(seed)  # PyTorch-Zufall setzen

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)  # falls GPU genutzt wird, auch dort den Seed setzen

    print(f"Seeds gesetzt auf: {seed}")  # kurzer Check


set_global_seeds(SEED)  # Seed-Funktion ausführen

Seeds gesetzt auf: 1


---

## Adapter-Wrapper für Stable-Baselines3

Der folgende Wrapper bildet die technische Schnittstelle zwischen dem Unity-ML-Agents-Wrapper und Stable-Baselines3. Er sorgt dafür, dass die Observations aus Unity als flacher Vektor vorliegen und im Gymnasium-Format an Stable-Baselines3 übergeben werden.

Das ist wichtig, weil der Agent mehrere Observation-Quellen nutzt. Für A2C und DQN mit `MlpPolicy` ist ein flacher Vektor die einfachste und stabilste Form der Eingabe.

In [5]:
class UnityFlatGymnasiumWrapper(gym.Env):
    metadata = {"render_modes": []}  # minimale Angabe für Gymnasium

    def __init__(self, unity_gym_env):
        super().__init__()  # Gymnasium-Basisklasse initialisieren

        self.unity_gym_env = unity_gym_env  # ursprüngliches Unity-Gym-Environment speichern

        self.action_space = spaces.Discrete(self.unity_gym_env.action_space.n)  # Action Space als Discrete übernehmen

        initial_obs = self.unity_gym_env.reset()  # einmal resetten, um die Observation-Struktur zu erkennen
        flat_obs = self._flatten_obs(initial_obs)  # Observation in einen flachen Vektor umwandeln

        self.observation_space = spaces.Box(
            low=-np.inf,  # keine feste untere Grenze setzen
            high=np.inf,  # keine feste obere Grenze setzen
            shape=flat_obs.shape,  # Dimension aus der ersten Observation ableiten
            dtype=np.float32  # Stable-Baselines3 arbeitet sauber mit float32
        )

    def _flatten_obs(self, obs: Any) -> np.ndarray:
        if isinstance(obs, tuple) and len(obs) == 2 and isinstance(obs[1], dict):
            obs = obs[0]  # falls reset schon Gymnasium-Format liefert, nur Observation nehmen

        if isinstance(obs, dict):
            parts = [np.asarray(value, dtype=np.float32).ravel() for value in obs.values()]  # Dict-Werte flatten
            return np.concatenate(parts).astype(np.float32)  # alles zu einem Vektor verbinden

        if isinstance(obs, (list, tuple)):
            parts = [np.asarray(part, dtype=np.float32).ravel() for part in obs]  # mehrere Observations flatten
            return np.concatenate(parts).astype(np.float32)  # alles zu einem Vektor verbinden

        return np.asarray(obs, dtype=np.float32).ravel()  # einzelne Observation flatten

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        if seed is not None:
            np.random.seed(seed)  # Seed für NumPy setzen, falls von Gymnasium übergeben

        obs = self.unity_gym_env.reset()  # Unity-Environment zurücksetzen
        flat_obs = self._flatten_obs(obs)  # Observation flatten

        return flat_obs, {}  # Gymnasium erwartet Observation und Info-Dict

    def step(self, action):
        action = int(action)  # Aktion sicher als int übergeben

        step_result = self.unity_gym_env.step(action)  # Aktion an Unity weitergeben

        if len(step_result) == 5:
            obs, reward, terminated, truncated, info = step_result  # falls bereits Gymnasium-Format
        else:
            obs, reward, done, info = step_result  # klassisches Gym-Format
            terminated = bool(done)  # done als terminated behandeln
            truncated = False  # truncated wird hier nicht separat unterschieden

        flat_obs = self._flatten_obs(obs)  # neue Observation flatten

        return flat_obs, float(reward), terminated, truncated, info  # Gymnasium-Step-Format zurückgeben

    def close(self):
        self.unity_gym_env.close()  # Unity-Gym-Environment sauber schließen

---

## Hilfsfunktion zum Erstellen des Unity-Environments

Damit A2C und DQN jeweils eine frische Unity-Verbindung bekommen, wird eine kleine Hilfsfunktion genutzt. Die Funktion verbindet Python mit Unity, erstellt den Unity-Gym-Wrapper und legt anschließend den eigenen Adapter-Wrapper darüber.

In [6]:
def make_unity_env(seed: int) -> UnityFlatGymnasiumWrapper:
    print("Starte Verbindung zu Unity. Jetzt in Unity Play drücken...")  # Hinweis für den Ablauf

    unity_env = UnityEnvironment(
        file_name=None,  # None bedeutet: Verbindung mit dem Unity Editor
        seed=seed,  # Seed an UnityEnvironment übergeben
        side_channels=[]  # aktuell keine zusätzlichen Side Channels
    )

    unity_gym_env = UnityToGymWrapper(
        unity_env,
        uint8_visual=False,  # keine visuellen Observations als uint8 verwenden
        flatten_branched=True,  # branched actions flatten
        allow_multiple_obs=True  # Vector Observation und Ray Observation gemeinsam erlauben
    )

    wrapped_env = UnityFlatGymnasiumWrapper(unity_gym_env)  # eigenes Gymnasium-kompatibles Environment erstellen

    print("Unity Environment verbunden.")  # Erfolgsmeldung
    print("Observation Space:", wrapped_env.observation_space)  # Beobachtungsraum anzeigen
    print("Action Space:", wrapped_env.action_space)  # Aktionsraum anzeigen

    return wrapped_env  # fertiges Environment zurückgeben

---

## Verbindung zum Unity-Environment herstellen

In dieser Zelle wird Python mit dem geöffneten Unity Editor verbunden. Erst diese Zelle starten und danach in Unity auf Play drücken.

In [7]:
env = make_unity_env(SEED)  # Unity-Environment für den Wrapper-Test erstellen

Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)


### Prüfung von Observation Space und Action Space

Hier wird geprüft, ob die Umgebung so erkannt wird, wie sie für A2C und DQN benötigt wird. Besonders wichtig ist, dass der Action Space als `Discrete(9)` erkannt wird und die Observations als flacher Box-Space vorliegen.

In [8]:
print("Observation Space:", env.observation_space)  # sollte ein flacher Box-Space sein
print("Action Space:", env.action_space)  # sollte Discrete(9) sein

assert isinstance(env.observation_space, spaces.Box), "Observation Space ist nicht Box. Bitte Wrapper prüfen."
assert isinstance(env.action_space, spaces.Discrete), "Action Space ist nicht Discrete. Bitte Unity Behavior Parameters prüfen."
assert env.action_space.n == 9, "Action Space ist nicht Discrete(9). Bitte Unity Behavior Parameters prüfen."

print("Spaces sind korrekt für Stable-Baselines3.")  # Bestätigung

Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
Spaces sind korrekt für Stable-Baselines3.


### Smoke-Test mit zufälligen Aktionen

Bevor A2C oder DQN trainiert werden, wird das Environment kurz mit zufälligen Aktionen getestet. Dadurch wird geprüft, ob `reset()` und `step()` grundsätzlich funktionieren und ob Unity korrekt mit Python kommuniziert.

In [9]:
obs, info = env.reset()  # Environment zurücksetzen und erste Observation holen

print("Observation Shape:", obs.shape)  # Form der Observation prüfen
print("Erste Observation-Auszüge:", obs[:10])  # nur erste Werte anzeigen, damit der Output lesbar bleibt

for i in range(10):
    action = env.action_space.sample()  # zufällige Aktion aus dem Action Space ziehen
    obs, reward, terminated, truncated, info = env.step(action)  # Aktion im Environment ausführen

    done = terminated or truncated  # Episode gilt als beendet, wenn terminated oder truncated true ist

    print(f"Step {i + 1}: action={action}, reward={reward}, done={done}")  # Schrittinfo anzeigen

    if done:
        obs, info = env.reset()  # wenn Episode endet, Environment zurücksetzen
        print("Episode beendet und Environment zurückgesetzt.")  # kurzer Hinweis

Observation Shape: (48,)
Erste Observation-Auszüge: [0. 0. 1. 1. 0. 0. 1. 1. 0. 0.]
Step 1: action=6, reward=-0.0010000000474974513, done=False
Step 2: action=8, reward=-0.0010000000474974513, done=False
Step 3: action=8, reward=-0.0010000000474974513, done=False
Step 4: action=8, reward=-0.0010000000474974513, done=False
Step 5: action=0, reward=-0.0010000000474974513, done=False
Step 6: action=6, reward=-0.0010000000474974513, done=False
Step 7: action=0, reward=-0.0010000000474974513, done=False
Step 8: action=4, reward=-0.0010000000474974513, done=False
Step 9: action=4, reward=-0.0010000000474974513, done=False
Step 10: action=4, reward=-0.0010000000474974513, done=False


---

### A2C Mini-Test vorbereiten

In diesem Abschnitt wird ein A2C-Modell erstellt. Der Mini-Test ist noch kein finales Training, sondern dient nur dazu zu prüfen, ob Stable-Baselines3 mit dem Unity-Wrapper trainieren kann.

In [10]:
a2c_log_dir_rel = Path("training") / "logs" / "A2C_Env1_Seed1_Test"  # relativer TensorBoard-Pfad für SB3
a2c_log_dir_abs = PROJECT_ROOT / a2c_log_dir_rel  # absoluter Pfad nur intern

a2c_model_dir = MODEL_DIR / "A2C_Env1_Seed1_Test"  # Modellordner für den A2C-Test

a2c_log_dir_abs.mkdir(parents=True, exist_ok=True)  # Logordner erstellen, falls er fehlt
a2c_model_dir.mkdir(parents=True, exist_ok=True)  # Modellordner erstellen, falls er fehlt

a2c_model = A2C(
    policy="MlpPolicy",  # MLP passt zu den flachen Vektor-Observations
    env=env,  # eigenes Gymnasium-kompatibles Unity-Environment
    learning_rate=7e-4,  # typischer A2C-Startwert
    n_steps=5,  # kurze Update-Horizonte, typisch für A2C
    gamma=0.99,  # Discount Factor
    ent_coef=0.01,  # kleiner Exploration-Anreiz
    vf_coef=0.5,  # Gewichtung des Value-Loss
    max_grad_norm=0.5,  # Gradient Clipping für stabileres Training
    seed=SEED,  # Seed für SB3
    verbose=0,  # verhindert ausführliche SB3-Ausgaben
    tensorboard_log=str(a2c_log_dir_rel)  # relativer Pfad, damit kein lokaler Windows-Pfad ausgegeben wird
)

print("A2C-Modell erstellt.")  # kurzer Check
print("TensorBoard Log Dir:", show_project_path(a2c_log_dir_abs))  # Pfad sauber relativ zum Projekt anzeigen

A2C-Modell erstellt.
TensorBoard Log Dir: rl_navigation_project/training/logs/A2C_Env1_Seed1_Test


---

### A2C Mini-Test ausführen

Jetzt wird A2C für wenige Timesteps trainiert. Wenn diese Zelle ohne Fehler durchläuft, ist die A2C-Pipeline technisch grundsätzlich funktionsfähig.

In [11]:
a2c_model.learn(
    total_timesteps=TEST_TIMESTEPS,  # kurzer technischer Testlauf
    tb_log_name="A2C_Env1_Seed1_Test",  # sauberer TensorBoard-Runname
    progress_bar=False  # keine zusätzlichen tqdm/rich-Abhängigkeiten
)

a2c_model.save(a2c_model_dir / "final_model")  # Testmodell speichern

print("A2C Mini-Test abgeschlossen und Modell gespeichert.")  # Erfolgsmeldung
print("Model Dir:", show_project_path(a2c_model_dir))  # Modellpfad sauber relativ zum Projekt anzeigen

A2C Mini-Test abgeschlossen und Modell gespeichert.
Model Dir: rl_navigation_project/training/models/A2C_Env1_Seed1_Test


---

### A2C Environment schließen

Nach dem A2C-Test wird das Environment geschlossen. Das ist wichtig, weil Unity- und Python-Verbindungen sonst hängen bleiben können, wenn direkt danach ein weiterer Algorithmus gestartet wird.

In [12]:
env.close()  # Unity-Verbindung schließen
print("A2C Environment geschlossen.")  # kurzer Check

A2C Environment geschlossen.


---

### Neue Unity-Verbindung für DQN

Für den DQN-Test wird das Unity-Environment neu verbunden. Dadurch wird vermieden, dass alte Zustände aus dem vorherigen A2C-Test im Wrapper hängen bleiben.

Wichtig: Diese Zelle starten und dann in Unity wieder auf Play drücken.

In [13]:
env_dqn = make_unity_env(SEED)  # frisches Unity-Environment für DQN erstellen

Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)


---

### DQN Action Space prüfen

DQN benötigt einen einfachen diskreten Aktionsraum. Deshalb wird hier nochmal geprüft, ob der Wrapper `Discrete(9)` erkennt.

In [14]:
print("DQN Observation Space:", env_dqn.observation_space)  # sollte Box sein
print("DQN Action Space:", env_dqn.action_space)  # sollte Discrete(9) sein

assert isinstance(env_dqn.observation_space, spaces.Box), "Observation Space ist nicht Box. Bitte Wrapper prüfen."
assert isinstance(env_dqn.action_space, spaces.Discrete), "DQN benötigt einen diskreten Action Space."
assert env_dqn.action_space.n == 9, "DQN benötigt Discrete(9). Bitte Unity Behavior Parameters prüfen."

print("DQN Spaces sind korrekt.")  # Bestätigung

DQN Observation Space: Box(-inf, inf, (48,), float32)
DQN Action Space: Discrete(9)
DQN Spaces sind korrekt.


---

### DQN Mini-Test vorbereiten

In diesem Abschnitt wird ein DQN-Modell erstellt. Auch dieser Mini-Test dient nur zur technischen Prüfung und ist noch kein finales Training.

Auch hier bleibt TensorBoard aktiv. Für `tensorboard_log` wird wieder ein relativer Pfad ab dem Projektroot verwendet, damit keine lokalen Windows-Pfade im Notebook-Output erscheinen.

In [15]:
dqn_log_dir_rel = Path("training") / "logs" / "DQN_Env1_Seed1_Test"  # relativer TensorBoard-Pfad für SB3
dqn_log_dir_abs = PROJECT_ROOT / dqn_log_dir_rel  # absoluter Pfad nur intern

dqn_model_dir = MODEL_DIR / "DQN_Env1_Seed1_Test"  # Modellordner für den DQN-Test

dqn_log_dir_abs.mkdir(parents=True, exist_ok=True)  # Logordner erstellen, falls er fehlt
dqn_model_dir.mkdir(parents=True, exist_ok=True)  # Modellordner erstellen, falls er fehlt

dqn_model = DQN(
    policy="MlpPolicy",  # MLP für flache Vektor-Observations
    env=env_dqn,  # eigenes Gymnasium-kompatibles Unity-Environment
    learning_rate=1e-4,  # typischer DQN-Startwert
    buffer_size=50_000,  # Replay Buffer für Erfahrungen
    learning_starts=1_000,  # erst nach 1000 Steps mit Updates starten
    batch_size=32,  # Batchgröße für Updates
    gamma=0.99,  # Discount Factor
    exploration_fraction=0.2,  # Anteil der Trainingszeit mit abnehmender Exploration
    exploration_initial_eps=1.0,  # startet komplett explorativ
    exploration_final_eps=0.05,  # minimale Exploration am Ende
    target_update_interval=1_000,  # Target Network regelmäßig aktualisieren
    seed=SEED,  # Seed für SB3
    verbose=0,  # verhindert ausführliche SB3-Ausgaben
    tensorboard_log=str(dqn_log_dir_rel)  # relativer Pfad, damit kein lokaler Windows-Pfad ausgegeben wird
)

print("DQN-Modell erstellt.")  # kurzer Check
print("TensorBoard Log Dir:", show_project_path(dqn_log_dir_abs))  # Pfad sauber relativ zum Projekt anzeigen

DQN-Modell erstellt.
TensorBoard Log Dir: rl_navigation_project/training/logs/DQN_Env1_Seed1_Test


---

### DQN Mini-Test ausführen

Jetzt wird DQN für wenige Timesteps trainiert. Wenn diese Zelle ohne Fehler durchläuft, ist auch die DQN-Pipeline technisch grundsätzlich funktionsfähig.

In [16]:
dqn_model.learn(
    total_timesteps=TEST_TIMESTEPS,  # kurzer technischer Testlauf
    tb_log_name="DQN_Env1_Seed1_Test",  # sauberer TensorBoard-Runname
    progress_bar=False  # keine zusätzlichen tqdm/rich-Abhängigkeiten
)

dqn_model.save(dqn_model_dir / "final_model")  # Testmodell speichern

print("DQN Mini-Test abgeschlossen und Modell gespeichert.")  # Erfolgsmeldung
print("Model Dir:", show_project_path(dqn_model_dir))  # Modellpfad sauber relativ zum Projekt anzeigen

DQN Mini-Test abgeschlossen und Modell gespeichert.
Model Dir: rl_navigation_project/training/models/DQN_Env1_Seed1_Test


---

### DQN Environment schließen

Nach dem DQN-Test wird auch diese Unity-Verbindung sauber geschlossen. Danach kann das Hyperparameter-Tuning vorbereitet werden.

In [17]:
env_dqn.close()  # DQN-Environment schließen
print("DQN Environment geschlossen.")  # kurzer Check

DQN Environment geschlossen.


---

### Prüfung der gespeicherten Testmodelle

Zum Abschluss wird geprüft, ob die Testmodelle in den erwarteten Modellordnern gespeichert wurden. Das ist ein einfacher technischer Check, ob Stable-Baselines3 die Modelle korrekt exportieren konnte.

In [18]:
expected_model_dirs = [
    MODEL_DIR / "A2C_Env1_Seed1_Test",  # erwarteter A2C-Testmodellordner
    MODEL_DIR / "DQN_Env1_Seed1_Test",  # erwarteter DQN-Testmodellordner
]

for model_dir in expected_model_dirs:
    print(show_project_path(model_dir), "exists:", model_dir.exists())  # prüfen, ob der Ordner existiert
    print("Files:", [file.name for file in model_dir.glob("*")])  # gespeicherte Dateien anzeigen

rl_navigation_project/training/models/A2C_Env1_Seed1_Test exists: True
Files: ['final_model.zip']
rl_navigation_project/training/models/DQN_Env1_Seed1_Test exists: True
Files: ['final_model.zip']


---

### Ergebnis des Wrapper-Tests

Der Stable-Baselines3 Wrapper-Test wurde erfolgreich abgeschlossen. Sowohl A2C als auch DQN konnten über den eigenen Adapter mit dem Unity-Environment verbunden und für einen kurzen technischen Testlauf trainiert werden.

Die Testmodelle wurden erfolgreich gespeichert:

- `rl_navigation_project/training/models/A2C_Env1_Seed1_Test/final_model.zip`
- `rl_navigation_project/training/models/DQN_Env1_Seed1_Test/final_model.zip`

Damit ist bestätigt, dass die technische Pipeline für A2C und DQN grundsätzlich funktioniert. Das Unity-Environment kann über Python angesteuert werden, die Observations werden als flacher Vektor verarbeitet und der diskrete Aktionsraum `Discrete(9)` wird korrekt genutzt.

Der nächste Schritt ist das Hyperparameter-Tuning mit Optuna. Dabei werden A2C und DQN auf `Env1_Simple` mit mehreren Parameterkombinationen getestet, bevor anschließend die finalen Trainingsruns durchgeführt werden.

## Zwischenfazit

Der Wrapper-Test zeigt, ob A2C und DQN technisch über Stable-Baselines3 mit dem Unity-Environment verbunden werden können. Dafür wurde ein eigener Gymnasium-kompatibler Adapter genutzt, der die Unity-Observations zu einem flachen Vektor zusammenführt.

Wichtig ist dabei, dass das Unity-Environment selbst nicht verändert wird. Der Adapter betrifft nur die Python-Schnittstelle zwischen Unity und Stable-Baselines3. Damit bleiben Bewegungslogik, Rewards, Observations und Action Space im Unity-Projekt konsistent.

Wenn beide Mini-Testläufe erfolgreich abgeschlossen wurden, ist die Grundlage für das anschließende Hyperparameter-Tuning gelegt. Die finalen Trainingsläufe werden in einem späteren Notebook durchgeführt. Dieses Notebook dient nur dazu, die technische Verbindung und grundlegende Trainierbarkeit zu validieren.